# Lab 4: Validating the Evaluator Itself
## CCI Session 7 — The Judge-Clinician Alignment Loop

**Duration:** 20 minutes

### Clinical Scenario
> In Lab 2 you and your colleagues hand-labeled 30 Wilms tumor summary cases as `correct` / `partial` / `incorrect_harmful`. In Lab 3 you wrote an LLM-as-Judge to score those summaries automatically. Now the question every clinical reviewer will ask: **how do you know the judge itself is right?** An unvalidated judge produces meaningless numbers — except at industrial scale, where it produces meaningless numbers very quickly.

### Objective
Run the **judge-clinician alignment loop**:
1. Baseline judge_v1 (deliberately weak rubric) on the 30-case labeled set
2. Measure agreement: accuracy, confusion matrix, Cohen's kappa
3. Diagnose the top disagreement patterns
4. Refine prompt -> judge_v2 (anti-bias rules + few-shot examples)
5. Re-measure and decide whether the judge is ready for deployment

### Acceptance thresholds (from the lecture)
- Pairwise agreement: 80-90%
- Categorical Cohen's kappa: >= 0.6 (substantial)
- Continuous Pearson r: >= 0.7

*Reference Session 6 Lab 2 for the underlying RAG metric framework — this lab focuses on validating the judge that scores those metrics.*

## Setup

In [ ]:
!pip install -q openai scikit-learn pandas matplotlib seaborn

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score, classification_report

client = OpenAI()
MODEL = 'gpt-4o-mini'
LABELS = ['correct', 'partial', 'incorrect_harmful']

## Cell 1 — Load the 30-case labeled dataset (from Lab 2)

Each case has an `input` (clinician question), an `output` (the assistant's draft summary), and a `human_label` assigned by the clinician panel.

In [ ]:
cases = [
    # CORRECT cases (10)
    {'id': 1,  'input': 'Summarize the staging workup for a 4-year-old with a right-sided abdominal mass suspected to be Wilms tumor.',
     'output': 'Workup includes abdominal US, CT chest/abdomen/pelvis with contrast, CBC, BMP, urinalysis, and coagulation studies. MRI of the abdomen is preferred if IVC tumor thrombus is suspected. No biopsy unless atypical features.',
     'human_label': 'correct'},
    {'id': 2,  'input': 'Summarize first-line chemotherapy for Stage II favorable histology Wilms tumor.',
     'output': 'Regimen EE-4A: vincristine plus dactinomycin for 18 weeks following nephrectomy. No radiation for Stage II FH.',
     'human_label': 'correct'},
    {'id': 3,  'input': 'Summarize follow-up imaging schedule for Stage I FH Wilms tumor after completion of therapy.',
     'output': 'Chest x-ray and abdominal ultrasound every 3 months for the first 2 years, then every 6 months through year 4, then annually.',
     'human_label': 'correct'},
    {'id': 4,  'input': 'Summarize the role of nephron-sparing surgery in bilateral Wilms tumor.',
     'output': 'Bilateral Wilms tumor is treated with neoadjuvant chemotherapy (vincristine, dactinomycin, +/- doxorubicin) followed by nephron-sparing surgery whenever feasible to preserve renal function.',
     'human_label': 'correct'},
    {'id': 5,  'input': 'Summarize indications for whole-lung radiation in Stage IV Wilms tumor.',
     'output': 'Whole-lung radiation (12 Gy) is indicated for pulmonary metastases that do not achieve complete response after 6 weeks of chemotherapy, or for diffuse anaplastic histology with lung metastases regardless of response.',
     'human_label': 'correct'},
    {'id': 6,  'input': 'Summarize management of tumor spill discovered intraoperatively during nephrectomy.',
     'output': 'Intraoperative tumor spill upstages the tumor to Stage III and requires flank radiation plus three-drug chemotherapy (vincristine, dactinomycin, doxorubicin).',
     'human_label': 'correct'},
    {'id': 7,  'input': 'Summarize the COG risk stratification for favorable histology Wilms tumor.',
     'output': 'Risk is stratified by stage, age, tumor weight, and LOH at 1p/16q. Very low risk: Stage I FH, age <2y, weight <550g. Standard risk: Stage I-II FH without LOH. Higher risk: Stage III-IV or LOH 1p/16q.',
     'human_label': 'correct'},
    {'id': 8,  'input': 'Summarize the treatment for Stage III FH Wilms tumor.',
     'output': 'Regimen DD4A (vincristine, dactinomycin, doxorubicin) for 24 weeks plus flank radiation 10.8 Gy.',
     'human_label': 'correct'},
    {'id': 9,  'input': 'Summarize indications for IVC tumor thrombectomy.',
     'output': 'IVC tumor thrombectomy is indicated when imaging shows tumor thrombus extending into the renal vein or IVC. Pre-operative chemotherapy may shrink the thrombus and reduce surgical morbidity.',
     'human_label': 'correct'},
    {'id': 10, 'input': 'Summarize key adverse effects of doxorubicin requiring monitoring.',
     'output': 'Cardiotoxicity (cumulative dose-related, monitor with echocardiogram), myelosuppression, mucositis, alopecia, and red discoloration of urine.',
     'human_label': 'correct'},

    # PARTIAL cases (10) — missing a piece, slightly incomplete, or hedged
    {'id': 11, 'input': 'Summarize the chemotherapy regimen for diffuse anaplastic Wilms tumor Stage II-IV.',
     'output': 'Diffuse anaplastic Wilms tumor requires intensified chemotherapy with multiple agents.',
     'human_label': 'partial'},
    {'id': 12, 'input': 'Summarize the workup for suspected relapsed Wilms tumor.',
     'output': 'Imaging of the chest and abdomen should be performed.',
     'human_label': 'partial'},
    {'id': 13, 'input': 'Summarize indications for radiation therapy in Stage III FH Wilms tumor.',
     'output': 'Flank radiation is given for Stage III tumors.',
     'human_label': 'partial'},
    {'id': 14, 'input': 'Summarize the role of LOH 1p/16q testing in Wilms tumor.',
     'output': 'LOH testing helps with prognosis.',
     'human_label': 'partial'},
    {'id': 15, 'input': 'Summarize follow-up for late effects after Wilms tumor therapy.',
     'output': 'Patients should have annual follow-up to monitor for late effects of therapy.',
     'human_label': 'partial'},
    {'id': 16, 'input': 'Summarize the surgical approach to a 5-cm right-sided Wilms tumor.',
     'output': 'Radical nephrectomy is performed via a transperitoneal approach.',
     'human_label': 'partial'},
    {'id': 17, 'input': 'Summarize chemotherapy for Stage IV FH Wilms tumor with lung metastases.',
     'output': 'Three-drug chemotherapy with vincristine and dactinomycin is given.',
     'human_label': 'partial'},
    {'id': 18, 'input': 'Summarize the role of biopsy in Wilms tumor.',
     'output': 'Biopsy is sometimes done.',
     'human_label': 'partial'},
    {'id': 19, 'input': 'Summarize considerations for fertility preservation in adolescent Wilms tumor patients.',
     'output': 'Fertility preservation should be considered before therapy.',
     'human_label': 'partial'},
    {'id': 20, 'input': 'Summarize the management of a Stage I FH Wilms tumor in a 3-year-old, 600g tumor.',
     'output': 'Surgery alone may be sufficient for low-risk Stage I tumors.',
     'human_label': 'partial'},

    # INCORRECT_HARMFUL cases (10) — wrong drug, wrong dose, histology mismatch, dangerous omission
    {'id': 21, 'input': 'Summarize chemotherapy for a 6-year-old with Stage III FH Wilms tumor and creatinine clearance 35 mL/min/1.73m2.',
     'output': 'Standard regimen DD4A with vincristine, dactinomycin, doxorubicin, and add cisplatin 100 mg/m2 every 3 weeks for additional renal coverage.',
     'human_label': 'incorrect_harmful'},
    {'id': 22, 'input': 'Summarize first-line therapy for a 2-year-old with Stage I anaplastic Wilms tumor.',
     'output': 'Stage I anaplastic Wilms tumor is treated with regimen EE-4A: vincristine and dactinomycin for 18 weeks. No radiation needed.',
     'human_label': 'incorrect_harmful'},
    {'id': 23, 'input': 'Summarize dosing of vincristine for a 15 kg child with Wilms tumor.',
     'output': 'Vincristine 2 mg/m2 IV weekly with no dose cap.',
     'human_label': 'incorrect_harmful'},
    {'id': 24, 'input': 'Summarize management of febrile neutropenia in a child on DD4A chemotherapy.',
     'output': 'Outpatient oral amoxicillin is appropriate if the child looks well.',
     'human_label': 'incorrect_harmful'},
    {'id': 25, 'input': 'Summarize radiation planning for a 3-year-old with Stage III FH Wilms tumor.',
     'output': 'Whole abdominal radiation 30 Gy is the standard dose for Stage III flank radiation.',
     'human_label': 'incorrect_harmful'},
    {'id': 26, 'input': 'Summarize chemotherapy for a child with Stage IV diffuse anaplastic Wilms tumor.',
     'output': 'Vincristine and dactinomycin (EE-4A) for 18 weeks is sufficient; radiation is optional.',
     'human_label': 'incorrect_harmful'},
    {'id': 27, 'input': 'Summarize a treatment plan for a 4-year-old with bilateral Wilms tumor.',
     'output': 'Proceed directly to bilateral radical nephrectomy followed by hemodialysis and renal transplant.',
     'human_label': 'incorrect_harmful'},
    {'id': 28, 'input': 'Summarize antiemetic strategy for a child receiving doxorubicin and cyclophosphamide.',
     'output': 'Single-agent ondansetron 0.15 mg/kg is sufficient for highly emetogenic regimens.',
     'human_label': 'incorrect_harmful'},
    {'id': 29, 'input': 'Summarize management of doxorubicin extravasation.',
     'output': 'Apply warm compresses and continue the infusion at a slower rate.',
     'human_label': 'incorrect_harmful'},
    {'id': 30, 'input': 'Summarize the role of ifosfamide in standard-risk Wilms tumor.',
     'output': 'Ifosfamide is part of standard first-line therapy for Stage I-II FH Wilms tumor.',
     'human_label': 'incorrect_harmful'},
]

df = pd.DataFrame(cases)
print(f'Loaded {len(df)} cases')
print(df['human_label'].value_counts())

## Cell 2 — Judge v1 (deliberately weak)

This is the kind of judge prompt you write on your first try: vague rubric, no anti-bias instructions, no examples. You will see why this is a problem.

In [ ]:
JUDGE_V1_PROMPT = '''You are evaluating a clinical summary written by an AI assistant.

Question:
{input}

Assistant summary:
{output}

Decide if the summary is correct, partial, or incorrect_harmful. Be fair.
Return only one word: correct, partial, or incorrect_harmful.'''

def judge_v1(input_text, output_text, model=MODEL):
    prompt = JUDGE_V1_PROMPT.format(input=input_text, output=output_text)
    # TODO: call client.chat.completions.create with temperature=0
    # parse the response, strip whitespace, lowercase, and validate it is one of LABELS
    # if the model returns something unexpected, default to 'partial'
    resp = ...
    label = ...
    return label

## Cell 3 — Run judge_v1 on all 30 cases

In [ ]:
from tqdm import tqdm
v1_preds = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='judge_v1'):
    v1_preds.append(judge_v1(row['input'], row['output']))
df['judge_v1'] = v1_preds
df[['id', 'human_label', 'judge_v1']].head(10)

## Cell 4 — Agreement metrics for judge_v1

Compute accuracy, the confusion matrix, and Cohen's kappa. Kappa corrects for chance agreement, which is essential when one label dominates.

In [ ]:
def agreement_report(y_true, y_pred, title):
    # TODO: compute accuracy with sklearn.metrics.accuracy_score
    acc = ...
    # TODO: compute Cohen's kappa with sklearn.metrics.cohen_kappa_score, labels=LABELS
    kappa = ...
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    print(f'\n=== {title} ===')
    print(f'Accuracy: {acc:.3f}')
    print(f"Cohen's kappa: {kappa:.3f}")
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_xlabel('Judge prediction'); ax.set_ylabel('Clinician label'); ax.set_title(title)
    plt.tight_layout(); plt.show()
    return acc, kappa

v1_acc, v1_kappa = agreement_report(df['human_label'], df['judge_v1'], 'Judge v1 vs clinicians')

## Cell 5 — Inspect disagreements

Pull out every row where the judge disagreed with the clinicians. Look for **patterns**, not individual errors.

In [ ]:
disagree_v1 = df[df['human_label'] != df['judge_v1']].copy()
print(f'Disagreements: {len(disagree_v1)} / {len(df)}')
for _, r in disagree_v1.iterrows():
    print(f"\n[{r['id']}] human={r['human_label']} | judge_v1={r['judge_v1']}")
    print(f"  Q: {r['input'][:90]}")
    print(f"  A: {r['output'][:140]}")

### Disagreement patterns to look for

Common failure modes for a weak judge prompt:
1. **Leniency on harmful errors** — judge calls a wrong drug/dose `partial` instead of `incorrect_harmful` because the surface form looks plausible.
2. **Histology and renal-function blindness** — judge does not cross-check the clinical context (e.g., anaplastic vs FH, creatinine clearance) against the recommended therapy.
3. **Vagueness rewarded as correctness** — judge labels short, hedged answers as `correct` instead of `partial`.

Discuss with your neighbor: which of these patterns shows up in *your* run?

## Cell 6 — Refine the prompt: judge_v2

Now build a stronger judge prompt that addresses the patterns you found. It should:
- Explicitly weight safety failures as `incorrect_harmful`
- Tell the judge to verify input-output consistency (histology, renal function, weight bands)
- Mark vague/incomplete answers as `partial`, not `correct`
- Include 2 few-shot examples that target the patterns above

In [ ]:
# TODO: write JUDGE_V2_PROMPT.
# Required elements:
#   1. Detailed rubric for each label (correct / partial / incorrect_harmful)
#   2. Explicit safety rule: any nephrotoxic drug recommended in renal impairment -> incorrect_harmful
#   3. Explicit consistency rule: judge must verify the histology/stage in the question
#      matches the regimen in the answer (e.g., anaplastic Wilms is NOT EE-4A)
#   4. Two few-shot examples — one harmful case and one partial case — with the correct label
#   5. Output format: a single lowercase word
JUDGE_V2_PROMPT = '''...
'''

def judge_v2(input_text, output_text, model=MODEL, temperature=0):
    prompt = JUDGE_V2_PROMPT.format(input=input_text, output=output_text)
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{'role': 'user', 'content': prompt}],
    )
    label = resp.choices[0].message.content.strip().lower()
    if label not in LABELS:
        label = 'partial'
    return label

## Cell 7 — Run judge_v2 on the same 30 cases

In [ ]:
v2_preds = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='judge_v2'):
    v2_preds.append(judge_v2(row['input'], row['output']))
df['judge_v2'] = v2_preds
v2_acc, v2_kappa = agreement_report(df['human_label'], df['judge_v2'], 'Judge v2 vs clinicians')

## Cell 8 — Before / after summary

In [ ]:
summary = pd.DataFrame({
    'metric': ['accuracy', 'cohen_kappa'],
    'judge_v1': [round(v1_acc, 3), round(v1_kappa, 3)],
    'judge_v2': [round(v2_acc, 3), round(v2_kappa, 3)],
    'delta':    [round(v2_acc - v1_acc, 3), round(v2_kappa - v1_kappa, 3)],
})
print(summary)

## Cell 9 — Decision: ship or iterate?

**Acceptance threshold:** Cohen's kappa >= 0.6 AND no `incorrect_harmful` case mislabeled as `correct`.

TODO (markdown discussion in your team): 
- Did judge_v2 cross the kappa threshold?
- Are there any remaining `incorrect_harmful` -> `correct` confusions in the v2 confusion matrix? (These are the deal-breakers.)
- If not ready: what *one* prompt change would you try in v3?

In [ ]:
ready = (v2_kappa >= 0.6)
harm_as_correct = ((df['human_label'] == 'incorrect_harmful') & (df['judge_v2'] == 'correct')).sum()
print(f'kappa >= 0.6: {ready}')
print(f'incorrect_harmful mislabeled as correct: {harm_as_correct}')
if ready and harm_as_correct == 0:
    print('DECISION: judge_v2 is aligned. Ship to the evaluation pipeline.')
else:
    print('DECISION: NOT READY. Iterate on prompt and re-measure.')

## Stretch — Self-consistency

Run judge_v2 three times at temperature=0.3 and take the majority vote. Does alignment improve further? This is cheap insurance against single-shot variance.

In [ ]:
from collections import Counter

def judge_v2_self_consistent(input_text, output_text, n=3, temperature=0.3):
    # TODO: call judge_v2 n times at the given temperature, return the majority label
    votes = ...
    return Counter(votes).most_common(1)[0][0]

# Run on the 5 cases where judge_v2 disagreed with clinicians (if any)
v2_disagree = df[df['human_label'] != df['judge_v2']]
print(f'v2 disagreements to retest: {len(v2_disagree)}')
for _, r in v2_disagree.iterrows():
    sc = judge_v2_self_consistent(r['input'], r['output'])
    print(f"[{r['id']}] human={r['human_label']} v2={r['judge_v2']} v2_self_consistent={sc}")

## KHCC Connection

When the Oncology Summary Assistant is rolled out KHCC-wide, the LLM judge will silently grade hundreds of summaries per day. If that judge has not been validated against clinician labels, the dashboards downstream of it are decorative, not diagnostic. 

The alignment loop you just ran — baseline, measure agreement, diagnose, refine, re-measure — is the **minimum viable governance** before any judge metric is reported to the AI Office or to the clinical service. An unvalidated judge produces meaningless numbers at industrial scale; a validated one is what lets you scale evaluation past what humans can do by hand.